In [1]:
import os
os.environ["HF_HOME"] = "E:/hf_cache"

from transformers import CLIPProcessor, RobertaTokenizer
from torch.utils.data import Dataset
from PIL import Image
import torch


class NewsBiasDataset(Dataset):
    def __init__(self, df,
                 clip_name="openai/clip-vit-base-patch32",
                 text_model_name="roberta-large",
                 max_text_len=512):
        self.df = df.reset_index(drop=True)

        self.clip_processor = CLIPProcessor.from_pretrained(clip_name)
        self.text_tokenizer = RobertaTokenizer.from_pretrained(text_model_name)
        self.max_text_len = max_text_len

        self.presence_cols = [
            "V1_Salience.present",
            "V2_Perspective.present",
            "V3_Color_Lighting.present",
            "V4_Symbolism.present",
            "T1_Loaded_Language.present",
            "T2_Moral_Judgment.present",
            "J1_Role_Framing.present",
            "J2_Selective_Imbalance.present",
            "J3_Stereotyping.present"
        ]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ----- image -----
        image = Image.open(row["image_path"]).convert("RGB")
        clip_inputs = self.clip_processor(
            images=image, return_tensors="pt"
        )

        # ----- long text -----
        with open(row["text_path"], "r", encoding="utf-8") as f:
            text = f.read()

        text_inputs = self.text_tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_text_len,
            return_tensors="pt"
        )

        presence = torch.tensor(
            row[self.presence_cols].values.astype(int),
            dtype=torch.float
        )

        relation = torch.tensor(int(row["D1_Relationship_Type"]), dtype=torch.long)

        return (
            clip_inputs["pixel_values"].squeeze(0),
            text_inputs["input_ids"].squeeze(0),
            text_inputs["attention_mask"].squeeze(0),
            presence,
            relation
        )


e:\GDELP\task_3\torch-gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import torch
import torch.nn as nn
from transformers import CLIPModel, RobertaModel
CLIP_NAME = "openai/clip-vit-base-patch32"
TEXT_MODEL_NAME = "roberta-large"

class MultimodalBiasModel(nn.Module):
    def __init__(self,
                 clip_name="openai/clip-vit-base-patch32",
                 text_model_name="roberta-large",
                 num_presence_labels=9,
                 num_relation_classes=5,
                 finetune_text=True):
        super().__init__()

        # ---- encoders ----
        self.clip = CLIPModel.from_pretrained(clip_name)
        self.text_encoder = RobertaModel.from_pretrained(text_model_name)

        for p in self.clip.parameters():
            p.requires_grad = False

        if not finetune_text:
            for p in self.text_encoder.parameters():
                p.requires_grad = False

        img_dim = self.clip.config.projection_dim          # 512
        txt_dim = self.text_encoder.config.hidden_size    # 1024 (large)

        joint_dim = img_dim + txt_dim

        # ---- heads ----
        self.presence_head = nn.Sequential(
            nn.Linear(joint_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_presence_labels)
        )

        self.relation_head = nn.Sequential(
            nn.Linear(joint_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_relation_classes)
        )

    def forward(self, pixel_values, input_ids, attention_mask):
        img_feat = self.clip.get_image_features(pixel_values=pixel_values)

        txt_outputs = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        txt_feat = txt_outputs.last_hidden_state[:, 0]  # [CLS]

        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)

        joint = torch.cat([img_feat, txt_feat], dim=1)

        return self.presence_head(joint), self.relation_head(joint)


In [3]:
import logging

logging.basicConfig(
    filename="training.log",
    filemode="w",
    format="%(asctime)s | %(levelname)s | %(message)s",
    level=logging.INFO
)



In [4]:
from tqdm import tqdm


def train_epoch(model, loader, optimizer, device, epoch):
    model.train()
    bce = nn.BCEWithLogitsLoss()
    ce = nn.CrossEntropyLoss()

    running_loss = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch}", ncols=100)

    for step, (pixel_values, input_ids, attention_mask, presence, relation) in enumerate(pbar):
        pixel_values = pixel_values.to(device)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        presence = presence.to(device)
        relation = relation.to(device)

        optimizer.zero_grad()

        p_logits, r_logits = model(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss_p = bce(p_logits, presence)
        loss_r = ce(r_logits, relation)
        loss = loss_p + loss_r

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if step % 10 == 0:
            pbar.set_postfix({
                "loss": f"{loss.item():.3f}",
                "Lp": f"{loss_p.item():.2f}",
                "Lr": f"{loss_r.item():.2f}"
            })

    avg_loss = running_loss / len(loader)
    logging.info(f"Epoch {epoch} | Train Loss: {avg_loss:.4f}")

    return avg_loss


In [5]:
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split


def train_model(df, save_path,
                batch_size=16,
                num_epochs=10,
                lr=1e-4,
                clip_model_name=CLIP_NAME):
    
    print(">>> ENTER train_model()")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    df = df.dropna(subset=["D1_Relationship_Type"])

    
    # -------- train / val split --------
    train_df, val_df = train_test_split(
        df,
        test_size=0.2,
        random_state=42,
        stratify=df["D1_Relationship_Type"]
    )

    train_set = NewsBiasDataset(train_df, clip_model_name)
    val_set = NewsBiasDataset(val_df, clip_model_name)

    train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True, num_workers=4
    )
    val_loader = DataLoader(
        val_set, batch_size=batch_size, shuffle=False, num_workers=4
    )

    # -------- model --------
    model = MultimodalBiasModel(
        clip_name=clip_model_name,   
        num_presence_labels=9,
        num_relation_classes=df["D1_Relationship_Type"].nunique()
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    bce_loss = torch.nn.BCEWithLogitsLoss()
    ce_loss = torch.nn.CrossEntropyLoss()

    # -------- training loop --------
    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        model.train()
        total_loss = 0

        for pixel_values, input_ids, attention_mask, presence, relation in train_loader:
            pixel_values = pixel_values.to(device)
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            presence = presence.to(device)
            relation = relation.to(device)


            image = image.to(device)
            text = text.to(device)
            presence = presence.to(device)
            relation = relation.to(device)

            optimizer.zero_grad()

            p_logits, r_logits = model(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask
            )

            loss_p = bce_loss(p_logits, presence)
            loss_r = ce_loss(r_logits, relation)

            loss = loss_p + loss_r
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"[Epoch {epoch+1}/{num_epochs}] Train Loss: {avg_loss:.4f}")

    # -------- save model --------
    torch.save(model.state_dict(), save_path)
    print(f"Model saved to {save_path}")

    return model


In [ ]:
if __name__ == "__main__":
    import pandas as pd

    #  读取你的标注表
    df = pd.read_csv("label-gemini-flash-lite-2.5.csv")

    # 必须包含以下列：
    # image_path, text_path
    # V1_Salience.present ... J3_Stereotyping.present
    # D1_Relationship_Type

    #  执行训练
    model = train_model(
        df=df,
        save_path="multimodal_bias_model.pt",
        batch_size=16,
        num_epochs=10,
        lr=1e-4
    )


>>> ENTER train_model()


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Using device: cuda


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/10


65.49s - unexpected character after line continuation character (<string>, line 1)
Traceback (most recent call last):
  File "e:\GDELP\task_3\torch-gpu\Lib\site-packages\debugpy\_vendored\pydevd\_pydevd_bundle\pydevd_vars.py", line 629, in change_attr_expression
    value = eval(expression, frame.f_globals, frame.f_locals)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1
    news_id  overall_bias_intensity V1_Salience.present  V1_Salience.score  \0            1                     2.0               False                0.0   1            2                     3.0               False                0.0   2            3                     3.0               False                0.0   3            4                     2.0                True                1.0   4            5                     1.0                True                2.0   ...        ...                     ...                 ...                ...   22115    22127                